In [1]:
import sys
print(sys.executable)

c:\Users\84828\miniforge3\python.exe


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import joblib
import os
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import BorderlineSMOTE
#from imblearn.under_sampling import TomekLinks
from imblearn.under_sampling import RandomUnderSampler
from IPython.display import display


In [3]:
df = pd.read_csv("../data/raw/paysim.csv")

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
print("===== DATA SHAPE =====")
print(df.shape)

print("===== DATA INFO =====")
print(df.info())

print("===== DESCRIBE =====")
print(df.describe())

===== DATA SHAPE =====
(6362620, 11)
===== DATA INFO =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB
None
===== DESCRIBE =====
               step        amount  oldbalanceOrg  newbalanceOrig  \
count  6.362620e+06  6.362620e+06   6.362620e+06    6.362620e+06   
mean   2.433972e+02  1.798619e+05   8.338831e+05    8.551137e+05   
std    1.423320e+02  6.038582e+05   2.888243e+06    2.924049e+06   
min    1.000000e+00  0.000000e+00   0.000000e+00    0.000000e+00   
25%    1.560000e+02  

In [5]:
print("===== TARGET DISTRIBUTION =====")
display(df['isFraud'].value_counts())
display(df["isFraud"].value_counts(normalize=True) * 100)

===== TARGET DISTRIBUTION =====


isFraud
0    6354407
1       8213
Name: count, dtype: int64

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64

TIỀN XỬ LÝ 

MISSING - DUPLICATED

In [6]:
#Kiểm tra dữ liệu thiếu và trùng lắp
print("===== MISSING VALUES =====")
print(df.isnull().sum())

print("===== DUPLICATED VALUES =====")
print(df.duplicated().sum())

===== MISSING VALUES =====
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64
===== DUPLICATED VALUES =====
0


Xử lý lại một số cột 

In [7]:
df["isMerchant"] = df["nameDest"].str.startswith("M").astype(int)

In [8]:
df['day'] = df['step'] // 24

In [9]:
df['errorBalanceOrig'] = (
    df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
)

df['errorBalanceDest'] = (
    df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
)

# Có thể thêm biến kiểm tra giao dịch rút hết tiền
df["isOrigBalanceZeroAfter"] = (df["newbalanceOrig"] == 0).astype(int)

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,isMerchant,day,errorBalanceOrig,errorBalanceDest,isOrigBalanceZeroAfter
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,1,0,0.0,9839.64,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,1,0,0.0,1864.28,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,0,0,0.0,181.00,1
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,0,0,0.0,21363.00,1
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,1,0,0.0,11668.14,0


Loại bỏ cột không cần thiết 

In [10]:
df1 = df.drop(['nameOrig', 'nameDest', "isFlaggedFraud"], axis=1)

Mã hóa biến phân loại 

In [11]:
df2 = pd.get_dummies(
    df1,
    columns=["type"],
    drop_first=False
)

dummy_cols = [col for col in df2.columns if col.startswith("type_")]
df2[dummy_cols] = df2[dummy_cols].astype(int)

df2["isMerchant"] = df2["isMerchant"].astype(int)
df2["isOrigBalanceZeroAfter"] = df2["isOrigBalanceZeroAfter"].astype(int)
df2["isFraud"] = df2["isFraud"].astype(int)

In [12]:
df2.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isMerchant,day,errorBalanceOrig,errorBalanceDest,isOrigBalanceZeroAfter,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,1,0,0.0,9839.64,0,0,0,0,1,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,1,0,0.0,1864.28,0,0,0,0,1,0
2,1,181.00,181.0,0.00,0.0,0.0,1,0,0,0.0,181.00,1,0,0,0,0,1
3,1,181.00,181.0,0.00,21182.0,0.0,1,0,0,0.0,21363.00,1,0,1,0,0,0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,1,0,0.0,11668.14,0,0,0,0,1,0


Xử lý mất cân bằng dữ liệu

In [13]:
X = df2.drop('isFraud', axis=1)
y = df2['isFraud']

feature_names = list(X.columns)

print("Number of features:", len(feature_names))
print(feature_names)

Number of features: 16
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isMerchant', 'day', 'errorBalanceOrig', 'errorBalanceDest', 'isOrigBalanceZeroAfter', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True) * 100)


Train shape: (5090096, 16)
Test shape: (1272524, 16)

Train target distribution:
isFraud
0    99.870926
1     0.129074
Name: proportion, dtype: float64

Test target distribution:
isFraud
0    99.870887
1     0.129113
Name: proportion, dtype: float64


In [15]:
numeric_cols = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "day",
    "errorBalanceOrig",
    "errorBalanceDest"
]

binary_cols = [
    col for col in X_train.columns
    if col not in numeric_cols
]

print("Numeric columns:")
print(numeric_cols)

print("\nBinary / one-hot columns:")
print(binary_cols)

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(
    X_train[numeric_cols]
)

X_test_scaled[numeric_cols] = scaler.transform(
    X_test[numeric_cols]
)

Numeric columns:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'day', 'errorBalanceOrig', 'errorBalanceDest']

Binary / one-hot columns:
['isMerchant', 'isOrigBalanceZeroAfter', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [16]:
rus = RandomUnderSampler(
    sampling_strategy=0.4,
    random_state=42
)

X_train_under, y_train_under = rus.fit_resample(
    X_train_scaled,
    y_train
)

print("\nAfter RandomUnderSampler:")
print(pd.Series(y_train_under).value_counts())
print(pd.Series(y_train_under).value_counts(normalize=True) * 100)


After RandomUnderSampler:
isFraud
0    16425
1     6570
Name: count, dtype: int64
isFraud
0    71.428571
1    28.571429
Name: proportion, dtype: float64


smote = BorderlineSMOTE(
    kind='borderline-1',
    random_state=42
)

X_final, y_final = smote.fit_resample(
    X_under,
    y_under
)


print("\nClass distribution after SMOTE:")
print(pd.Series(y_final).value_counts())

print(pd.Series(y_final).value_counts(normalize=True) * 100)

Lưu dữ liệu

In [17]:

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../model/saved_models", exist_ok=True)

# Full processed data
df2.to_csv(
    "../data/processed/all_processed.csv",
    index=False
)

# Original scaled train
train_original_df = pd.concat(
    [
        X_train_scaled.reset_index(drop=True),
        y_train.reset_index(drop=True)
    ],
    axis=1
)

# Test scaled
test_df = pd.concat(
    [
        X_test_scaled.reset_index(drop=True),
        y_test.reset_index(drop=True)
    ],
    axis=1
)

# Undersampled train
train_under_df = pd.concat(
    [
        pd.DataFrame(X_train_under, columns=X_train_scaled.columns),
        pd.Series(y_train_under, name="isFraud")
    ],
    axis=1
)

train_original_df.to_csv(
    "../data/processed/train_original.csv",
    index=False
)

train_under_df.to_csv(
    "../data/processed/train_under.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test.csv",
    index=False
)

joblib.dump(
    scaler,
    "../model/saved_models/scaler.pkl"
)

joblib.dump(
    feature_names,
    "../model/saved_models/feature_names.pkl"
)

joblib.dump(
    numeric_cols,
    "../model/saved_models/numeric_cols.pkl"
)

joblib.dump(
    binary_cols,
    "../model/saved_models/binary_cols.pkl"
)

print("Saved successfully!")

Saved successfully!
